In [49]:
import numpy as np
import pandas as pd

import metric

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])
pd.set_option('display.max_columns', 50)
print(train.shape, test.shape, events.shape)

(11091, 5) (4909, 4) (328905, 14)


In [51]:
events_cl = events.drop_duplicates()
cookies_all = pd.concat([train.drop(columns="target"), test], ignore_index=True)

#берем только события в окне
events_cl = events_cl.merge(cookies_all, on='cookie_id', how='inner')
events_cl = events_cl[(events_cl['event_ts'] >= events_cl['window_start_ts']) & (events_cl['event_ts'] <= events_cl['window_end_ts'])]

events_cl = events_cl.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

events_cl['platform_norm'] = events_cl['platform'].str.lower() #исправляем разнорегистровые названия

In [52]:
import re

no_heading = re.compile(r'headless|phantom|puppeteer|selenium|playwright', re.I)
lib = re.compile(r'python|curl|scrapy|requests|node-fetch|axios|libwww|wget|go-http-client|java/|php', re.I)
android = re.compile(r'android', re.I)
ios = re.compile(r'iphone|ipad|cpu ios|ios\b', re.I)
desktop = re.compile(r'windows nt|macintosh|x11; linux', re.I)

ua = events_cl['user_agent'].fillna('')
ua_flags = pd.DataFrame(index=ua.index)
ua_flags['is_headless'] = ua.str.contains(no_heading)
ua_flags['is_lib_ua'] = ua.str.contains(lib)
ua_flags['ua_has_android'] = ua.str.contains(android)
ua_flags['ua_has_ios'] = ua.str.contains(ios)
ua_flags['ua_has_desktop_os'] = ua.str.contains(desktop)
ua_flags['is_official_app'] = ua.str.startswith('Avito/')

ua_flags['browser_family'] = np.select(
    [ua_flags['is_headless'], ua.str.contains('YaBrowser', case=False), ua_flags['is_official_app'],
     ua.str.contains('Firefox', case=False), ua.str.contains('Chrome', case=False), ua.str.contains('Safari', case=False)],
    ['headless', 'yabrowser', 'official_app', 'firefox', 'chrome', 'safari'], default='other')

ua_flags['os_family'] = np.select(
    [ua_flags['ua_has_android'], ua_flags['ua_has_ios'], ua.str.contains('Macintosh', case=False),
     ua.str.contains('Windows', case=False), ua.str.contains('X11; Linux', case=False)],
    ['android', 'ios', 'mac', 'windows', 'linux'], default='other')

events_cl = pd.concat([events_cl, ua_flags], axis=1)

print(events_cl['browser_family'].value_counts())

browser_family
chrome          141193
official_app     51372
firefox          37530
yabrowser        33083
headless         10283
other             5848
safari            4591
Name: count, dtype: int64


In [53]:
events_cl.shape

(283900, 26)

In [55]:
def platform_ua_mismatch(platform_norm, f):
    m = pd.Series(False, index=platform_norm.index)
    m |= platform_norm.eq('android') & ~f['ua_has_android']
    m |= platform_norm.isin(['ios', 'iphone']) & ~f['ua_has_ios']
    m |= platform_norm.isin(['desktop', 'web']) & ~f['ua_has_desktop_os']
    return m

events_cl['platform_ua_mismatch'] = platform_ua_mismatch(events_cl['platform_norm'], ua_flags)

In [56]:
print('доля событий с несогласованными platform и UA:', events_cl['platform_ua_mismatch'].mean().round(4))

доля событий с несогласованными platform и UA: 0.0206


In [57]:
events_cl.shape

(283900, 27)

In [35]:
def entropy(counts):
    # энтропия Шеннона
    counts = counts[counts > 0]
    if counts.sum() == 0:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

In [36]:
def timing_feats(s):
    ts = s.values.astype('datetime64[s]').astype(np.int64)
    if len(ts) < 2:
        return pd.Series({'span_sec': 0.0, 'gap_mean': np.nan, 'gap_std': np.nan,
                          'gap_min': np.nan, 'gap_median': np.nan, 'event_rate': np.nan})
    gaps = np.diff(ts)
    span = ts[-1] - ts[0]
    return pd.Series({'span_sec': float(span), 'gap_mean': gaps.mean(), 'gap_std': gaps.std(),
                      'gap_min': gaps.min(), 'gap_median': np.median(gaps),
                      'event_rate': len(ts) / (span + 1.0)})

In [59]:
EVENT_NAMES = events_cl['event_name'].unique()

def build_features(ev, cookies):
    g = ev.groupby('cookie_id', sort=False)
    feats = pd.DataFrame(index=pd.Index(cookies['cookie_id'], name='cookie_id'))

    #объём
    feats['n_events'] = g.size()
    feats['n_unique_event_names'] = g['event_name'].nunique()
    feats['n_unique_platforms'] = g['platform_norm'].nunique()
    feats['n_unique_ua'] = g['user_agent'].nunique()

    #типы событий: счётчик, доля, флаг наличия
    counts = pd.crosstab(ev['cookie_id'], ev['event_name']).reindex(columns=EVENT_NAMES, fill_value=0)
    counts = counts.reindex(feats.index, fill_value=0)
    for name in EVENT_NAMES:
        feats[f'cnt_{name}'] = counts[name]
        feats[f'share_{name}'] = counts[name] / feats['n_events'].replace(0, np.nan)
        feats[f'has_{name}'] = (counts[name] > 0).astype(int)

    #разнообразие контента
    feats['n_unique_item'] = g['item_id'].nunique()
    feats['n_unique_category'] = g['item_category'].nunique()
    feats['n_unique_location'] = g['item_location'].nunique()
    feats['n_unique_query'] = g['search_query'].nunique()
    feats['max_search_page'] = g['search_page'].max()
    feats['category_entropy'] = g['item_category'].apply(lambda s: entropy(s.value_counts().values))
    feats['location_entropy'] = g['item_location'].apply(lambda s: entropy(s.value_counts().values))

    #тайминги
    feats = feats.join(g['event_ts'].apply(timing_feats).unstack())
    hours = ev['event_ts'].dt.hour
    feats['frac_night_hours'] = hours.between(2, 5).groupby(ev['cookie_id']).mean()   # активность в 02:00-05:59
    feats['hour_std'] = hours.groupby(ev['cookie_id']).std()

    #курсор
    has_ptr = ev['pointer_x'].notna()
    feats['frac_events_with_pointer'] = has_ptr.groupby(ev['cookie_id']).mean()
    feats['pointer_xy_std'] = ev[has_ptr].groupby('cookie_id')[['pointer_x', 'pointer_y']].std().mean(axis=1)
    feats['has_pointer_data'] = (feats['frac_events_with_pointer'] > 0).astype(int)

    #UA / платформа
    feats['has_headless_ua'] = g['is_headless'].max().astype(int)
    feats['has_lib_ua'] = g['is_lib_ua'].max().astype(int)
    feats['frac_official_app_ua'] = g['is_official_app'].mean()
    feats['n_browser_family'] = g['browser_family'].nunique()
    feats['n_os_family'] = g['os_family'].nunique()
    feats['has_platform_ua_mismatch'] = g['platform_ua_mismatch'].max().astype(int)
    feats['frac_platform_ua_mismatch'] = g['platform_ua_mismatch'].mean()

    #признаки самого кука
    age = (cookies.set_index('cookie_id')['window_start_ts'] - cookies.set_index('cookie_id')['cookie_created_at'])
    feats['cookie_age_hours'] = age.dt.total_seconds() / 3600.0

    # где признак не определён (нет координат или одно событие) там 0
    # флаги has_* сохраняют информацию что нет данных
    return feats.fillna(0).reset_index()


features_all = build_features(events_cl, cookies_all)
print(features_all.shape)

(16000, 61)


In [60]:
features_all.shape

(16000, 61)

In [61]:
features_all.to_csv("features.csv", index=False)


In [63]:
features = pd.read_csv("features.csv")

In [67]:
train = train.merge(features, on='cookie_id').drop(columns=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = test.merge(features, on='cookie_id').drop(columns=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
y = train['target'].values
X = train.drop(columns=['cookie_id', 'target'])
# определяем и убираем константные признаки, тк они ничего не дают
const_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_cols)
X_test = test.drop(columns='cookie_id')

In [68]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import optuna

rs = 42
skf = StratifiedKFold(shuffle=True, random_state=rs)
folds = list(skf.split(X, y))

In [69]:
def cv_oof(params, seed=rs):
    oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        model = CatBoostClassifier(**params, loss_function='Logloss', eval_metric='PRAUC',
                              random_seed=seed, verbose=False, allow_writing_files=False)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    return oof

In [70]:
import metric

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1200, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 5.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 3.0),
        'border_count': trial.suggest_categorical('border_count', [64, 128, 254]),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'auto_class_weights': trial.suggest_categorical('auto_class_weights', ['None', 'SqrtBalanced', 'Balanced']),
        'grow_policy': 'Lossguide',
        'bootstrap_type': 'Bayesian',
        'thread_count': -1,
    }
    params['max_leaves'] = trial.suggest_int('max_leaves', 8, 64)
    if params['auto_class_weights'] == 'None':
        params['auto_class_weights'] = None

    oof = cv_oof(params)
    trial.set_user_attr('pr_auc', average_precision_score(y, oof))
    trial.set_user_attr('roc_auc', roc_auc_score(y, oof))
    return metric.precision_at_recall(y, oof)

In [71]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=rs))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print('лучший Precision@Recall', round(study.best_value, 4))
print('PR-AUC / ROC-AUC этой попытки:', round(study.best_trial.user_attrs['pr_auc'], 4), '/', round(study.best_trial.user_attrs['roc_auc'], 4))
print(study.best_params)

[I 2026-09-28 21:40:56,711] A new study created in memory with name: no-name-cc4fe9a0-a895-4843-bf1d-1cbeeaa27207
Best trial: 0. Best value: 0.691886:   2%|▎         | 1/40 [00:15<10:02, 15.44s/it]

[I 2026-09-28 21:41:12,155] Trial 0 finished with value: 0.6918859649122807 and parameters: {'iterations': 600, 'learning_rate': 0.19063571821788408, 'depth': 7, 'l2_leaf_reg': 18.361096041714063, 'random_strength': 0.864491338167939, 'bagging_temperature': 0.46798356100860794, 'border_count': 128, 'min_data_in_leaf': 36, 'auto_class_weights': 'SqrtBalanced', 'max_leaves': 20}. Best is trial 0 with value: 0.6918859649122807.


Best trial: 0. Best value: 0.691886:   5%|▌         | 2/40 [00:22<06:41, 10.57s/it]

[I 2026-09-28 21:41:19,320] Trial 1 finished with value: 0.6870229007633588 and parameters: {'iterations': 400, 'learning_rate': 0.044846856872152424, 'depth': 4, 'l2_leaf_reg': 16.2179365173349, 'random_strength': 2.2165305913463675, 'bagging_temperature': 0.8736874205941257, 'border_count': 64, 'min_data_in_leaf': 19, 'auto_class_weights': 'SqrtBalanced', 'max_leaves': 37}. Best is trial 0 with value: 0.6918859649122807.


Best trial: 0. Best value: 0.691886:   8%|▊         | 3/40 [00:42<09:13, 14.95s/it]

...

[I 2026-09-28 21:49:57,645] Trial 37 finished with value: 0.7026726057906458 and parameters: {'iterations': 600, 'learning_rate': 0.04038496416809666, 'depth': 6, 'l2_leaf_reg': 22.35570748832643, 'random_strength': 0.678285264371008, 'bagging_temperature': 1.417376629899437, 'border_count': 128, 'min_data_in_leaf': 35, 'auto_class_weights': 'SqrtBalanced', 'max_leaves': 15}. Best is trial 17 with value: 0.7126696832579186.


Best trial: 17. Best value: 0.71267:  98%|█████████▊| 39/40 [09:13<00:14, 14.04s/it]

[I 2026-09-28 21:50:09,759] Trial 38 finished with value: 0.6745182012847966 and parameters: {'iterations': 500, 'learning_rate': 0.07812356262478588, 'depth': 6, 'l2_leaf_reg': 18.12591061553961, 'random_strength': 0.6068911267951111, 'bagging_temperature': 0.8734211843272777, 'border_count': 128, 'min_data_in_leaf': 38, 'auto_class_weights': 'SqrtBalanced', 'max_leaves': 27}. Best is trial 17 with value: 0.7126696832579186.


Best trial: 17. Best value: 0.71267: 100%|██████████| 40/40 [09:27<00:00, 14.18s/it]

[I 2026-09-28 21:50:24,108] Trial 39 finished with value: 0.6796116504854369 and parameters: {'iterations': 600, 'learning_rate': 0.03624056898503797, 'depth': 7, 'l2_leaf_reg': 19.783394639431847, 'random_strength': 1.7858364201761865, 'bagging_temperature': 0.883877308620003, 'border_count': 128, 'min_data_in_leaf': 36, 'auto_class_weights': 'SqrtBalanced', 'max_leaves': 22}. Best is trial 17 with value: 0.7126696832579186.
лучший Precision@Recall 0.7127
PR-AUC / ROC-AUC этой попытки: 0.756 / 0.9217
{'iterations': 400, 'learning_rate': 0.07072695065067787, 'depth': 6, 'l2_leaf_reg': 16.120255691707413, 'random_strength': 0.5997096079684272, 'bagging_temperature': 1.0605742324061913, 'border_count': 128, 'min_data_in_leaf': 38, 'auto_class_weights': 'SqrtBalanced', 'max_leaves': 22}


In [76]:
best_params = dict(study.best_params)
best_params.pop('max_leaves')
best_params

{'iterations': 400,
 'learning_rate': 0.07072695065067787,
 'depth': 6,
 'l2_leaf_reg': 16.120255691707413,
 'random_strength': 0.5997096079684272,
 'bagging_temperature': 1.0605742324061913,
 'border_count': 128,
 'min_data_in_leaf': 38,
 'auto_class_weights': 'SqrtBalanced'}

In [77]:
m = CatBoostClassifier(**best_params, loss_function='Logloss', eval_metric='PRAUC',
                              random_seed=rs, verbose=False, allow_writing_files=False)
m.fit(X, y)

CatBoostClassifier(allow_writing_files=False, auto_class_weights='SqrtBalanced', bagging_temperature=1.0605742324061913, border_count=128, depth=6, eval_metric='PRAUC', iterations=400, l2_leaf_reg=16.120255691707413, learning_rate=0.07072695065067787, loss_function='Logloss', min_data_in_leaf=38, random_seed=42, random_strength=0.5997096079684272, verbose=False)

In [78]:
pred = m.predict_proba(X_test)[:, 1]

In [81]:
submission = pd.DataFrame({'cookie_id': test['cookie_id'], 'score': pred})
submission.to_csv('submission.csv', index=False)

In [80]:
submission.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.0
1,ck_a76ee3b3e3e522fd,0.0
2,ck_94c9a4d382689e82,0.0
3,ck_8eaf9509ad9462a0,0.0
4,ck_9a88a5a989cb5bc6,0.0
